### Import Library & Load Dataset
Import library yang dibutuhin terus load dataset IRIS.csv

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('IRIS.csv')
df.head()

### Cek Missing Value
Cek dulu ada data yang kosong atau engga di tiap kolom. Kalau ada nanti bisa di isi pake mean/median atau dihapus

In [ ]:
# cek ada data kosong atau engga
print(df.isnull().sum())

### Handling Duplikat
Cek data yang kembar/duplikat terus dihapus biar ga bias nanti pas training model

In [ ]:
print(f"Jumlah duplikat: {df.duplicated().sum()}")
print(df[df.duplicated(keep=False)]) # tampilin baris yg duplikat

df = df.drop_duplicates().reset_index(drop=True)
print(f"Setelah dihapus: {len(df)} baris")

### Deteksi & Penanganan Outlier
Pake boxplot buat liat kolom mana yang ada outliernya. Ternyata sepal_width ada outlier, jadi ditangani pake metode capping (nilai yg kelewat ekstrem dibatesin ke batas IQR)

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(data=df.select_dtypes(include='number'))
plt.title("Boxplot untuk Deteksi Outlier")
plt.show()

# dari boxplot kelihatan sepal_width ada outlier, jadi di-capping pake IQR
Q1 = df['sepal_width'].quantile(0.25)
Q3 = df['sepal_width'].quantile(0.75)
IQR = Q3 - Q1
batas_bawah = Q1 - 1.5 * IQR
batas_atas = Q3 + 1.5 * IQR

df['sepal_width'] = np.clip(df['sepal_width'], batas_bawah, batas_atas) # clip biar outliernya dibatesin

### Feature Engineering
Bikin 2 fitur baru tanpa nyentuh kolom target (species):
- `petal_area`: luas petal dari panjang x lebar
- `petal_size`: kategori ukuran petal (Kecil/Sedang/Besar) buat nanti di ordinal encoding

In [ ]:
df['petal_area'] = df['petal_length'] * df['petal_width'] # fitur baru: luas petal

# bagi jadi 3 kategori biar nanti bisa di ordinal encoding
df['petal_size'] = pd.qcut(df['petal_area'], q=3, labels=['Kecil', 'Sedang', 'Besar'])

df[['petal_length', 'petal_width', 'petal_area', 'petal_size']].head()

### Standarisasi Kolom Numerik
Pake StandardScaler biar semua kolom angka punya skala yg sama (mean=0, std=1). Ini penting supaya kolom yang nilainya gede ga terlalu dominan

In [ ]:
kolom_angka = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'petal_area']

scaler = StandardScaler()
df_scaled = df.copy() # copy dulu biar data asli ga ketimpa
df_scaled[kolom_angka] = scaler.fit_transform(df[kolom_angka])

print(df_scaled[kolom_angka].mean()) # harusnya deket 0
df_scaled[kolom_angka].head()

### Ordinal Encoding
Kolom `petal_size` punya urutan (Kecil < Sedang < Besar) jadi pake ordinal encoding biar urutannya ke-capture

In [ ]:
# petal_size ada urutannya jadi pake ordinal
urutan = {'Kecil': 0, 'Sedang': 1, 'Besar': 2}
df_scaled['petal_size_encoded'] = df_scaled['petal_size'].map(urutan)

### One-Hot Encoding
Kolom `species` itu nominal (ga ada urutannya), jadi pake one-hot encoding biar model ga salah ngira ada peringkat antar spesies

In [ ]:
df_final = pd.get_dummies(df_scaled, columns=['species'], prefix='species', dtype=int) # one-hot soalnya species ga ada urutannya
df_final.head()